In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("iceberg-hands-on")
    # pulls the Iceberg jar automatically from Maven — no manual download
    .config("spark.jars.packages",
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.9.1")
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.local.type", "hadoop")
    .config("spark.sql.catalog.local.warehouse", "file:///tmp/warehouse")
    .getOrCreate()
)

### Insert + read

In [3]:
%%sql
CREATE TABLE local.shop.orders (
    order_id   BIGINT,
    customer   STRING,
    product    STRING,
    amount     DOUBLE,
    order_ts   TIMESTAMP
)
USING iceberg
PARTITIONED BY (days(order_ts))

26/07/11 07:45:44 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


++
||
++
++

In [4]:
%%sql
INSERT INTO local.shop.orders VALUES
(1, 'ravi',  'keyboard', 2500.0,  TIMESTAMP '2026-07-01 09:15:00'),
 (2, 'sneha', 'monitor',  12000.0, TIMESTAMP '2026-07-01 14:30:00'),
 (3, 'arjun', 'mouse',    800.0,   TIMESTAMP '2026-07-02 10:05:00'),
 (4, 'priya', 'laptop',   65000.0, TIMESTAMP '2026-07-02 18:45:00'),
 (5, 'ravi',  'webcam',   3200.0,  TIMESTAMP '2026-07-03 11:20:00');



++
||
++
++

In [5]:
%%sql

SELECT * FROM local.shop.orders ORDER BY order_id;


order_id,customer,product,amount,order_ts
1,ravi,keyboard,2500.0,2026-07-01 09:15:00
2,sneha,monitor,12000.0,2026-07-01 14:30:00
3,arjun,mouse,800.0,2026-07-02 10:05:00
4,priya,laptop,65000.0,2026-07-02 18:45:00
5,ravi,webcam,3200.0,2026-07-03 11:20:00


### Hidden partitioning

In [6]:
%%sql

SELECT order_id, customer, product, amount
FROM local.shop.orders
WHERE order_ts BETWEEN TIMESTAMP '2026-07-02 00:00:00'
                   AND TIMESTAMP '2026-07-02 23:59:59';

order_id,customer,product,amount
3,arjun,mouse,800.0
4,priya,laptop,65000.0


In [8]:
%%sql

SELECT partition, record_count, file_count
FROM local.shop.orders.partitions ORDER BY partition;

partition,record_count,file_count
"Row(order_ts_day=datetime.date(2026, 7, 1))",2,1
"Row(order_ts_day=datetime.date(2026, 7, 2))",2,1
"Row(order_ts_day=datetime.date(2026, 7, 3))",1,1


On a Hive table, if your analyst forgets the partition column in the WHERE clause, you scan the entire table — at a company like Netflix that's a petabyte scan, real money. Here there IS no partition column to forget. Iceberg took my timestamp filter and pruned to just the 2026-07-02 files. Users just query. Iceberg figures out what to skip."

### Schema evolution

In [9]:
%%sql
    
ALTER TABLE local.shop.orders ADD COLUMN discount DOUBLE;


++
||
++
++

In [10]:
%%sql
ALTER TABLE local.shop.orders RENAME COLUMN amount TO total_amount;

++
||
++
++

In [11]:
%%sql

INSERT INTO local.shop.orders VALUES
 (6, 'meera', 'headset', 4500.0, TIMESTAMP '2026-07-03 16:10:00', 500.0);


++
||
++
++

In [12]:
%%sql

SELECT order_id, customer, total_amount, discount
FROM local.shop.orders ORDER BY order_id;

order_id,customer,total_amount,discount
1,ravi,2500.0,None
2,sneha,12000.0,None
3,arjun,800.0,None
4,priya,65000.0,None
5,ravi,3200.0,None
6,meera,4500.0,500.0


###  Snapshots: the table remembers everything

In [13]:
%%sql

SELECT snapshot_id, committed_at, operation,
       summary['added-records'] AS added
FROM local.shop.orders.snapshots ORDER BY committed_at;

snapshot_id,committed_at,operation,added
625913880589721533,2026-07-11 07:45:55.679000,append,5
6051262841189964756,2026-07-11 07:49:07.471000,append,1


### Time travel 

In [17]:
%%sql

-- the table exactly as it was after the FIRST insert:
SELECT *
FROM local.shop.orders VERSION AS OF 625913880589721533;

-- or by time:
-- SELECT * FROM local.shop.orders TIMESTAMP AS OF '2026-07-07 18:40:31';

order_id,customer,product,amount,order_ts
3,arjun,mouse,800.0,2026-07-02 10:05:00
4,priya,laptop,65000.0,2026-07-02 18:45:00
5,ravi,webcam,3200.0,2026-07-03 11:20:00
1,ravi,keyboard,2500.0,2026-07-01 09:15:00
2,sneha,monitor,12000.0,2026-07-01 14:30:00


### THE DISASTER + the undo button

In [18]:
%%sql

SELECT snapshot_id FROM local.shop.orders.snapshots
ORDER BY committed_at DESC LIMIT 1;

snapshot_id
6051262841189964756


In [19]:
%%sql
UPDATE local.shop.orders SET total_amount = 0;

++
||
++
++

In [20]:
%%sql
SELECT order_id, total_amount FROM local.shop.orders LIMIT 3;
-- every single amount: 0.0  😱

order_id,total_amount
3,0.0
4,0.0
5,0.0


In [21]:
%%sql
CALL local.system.rollback_to_snapshot('shop.orders', 6051262841189964756)


previous_snapshot_id,current_snapshot_id
2516236904944342853,6051262841189964756


In [26]:
%%sql
SELECT order_id, total_amount FROM local.shop.orders LIMIT 3;

order_id,total_amount
3,800.0
4,65000.0
1,2500.0
2,12000.0
6,4500.0
5,2900.0
7,5500.0


In [3]:
### MERGE INTO: warehouse-grade upserts

In [23]:
%%sql
CREATE OR REPLACE TEMPORARY VIEW updates AS
SELECT * FROM VALUES
 (5, 'ravi',  'webcam', 2900.0, TIMESTAMP '2026-07-03 11:20:00', 300.0),
 (7, 'kiran', 'ssd',    5500.0, TIMESTAMP '2026-07-04 09:00:00', 0.0)
AS t(order_id, customer, product, total_amount, order_ts, discount);


++
||
++
++

In [24]:
%%sql
MERGE INTO local.shop.orders o
USING updates u ON o.order_id = u.order_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

++
||
++
++

### Partition evolution 

In [27]:
%%sql
ALTER TABLE local.shop.orders ADD PARTITION FIELD bucket(4, customer)


++
||
++
++

In [28]:
%%sql

INSERT INTO local.shop.orders VALUES
 (8, 'anita', 'gpu', 95000.0, TIMESTAMP '2026-07-05 12:00:00', 0.0);


++
||
++
++

In [29]:
%%sql
SELECT partition, record_count FROM local.shop.orders.partitions ORDER BY partition;

partition,record_count
"Row(order_ts_day=datetime.date(2026, 7, 1), customer_bucket_4=None)",2
"Row(order_ts_day=datetime.date(2026, 7, 2), customer_bucket_4=None)",2
"Row(order_ts_day=datetime.date(2026, 7, 3), customer_bucket_4=None)",2
"Row(order_ts_day=datetime.date(2026, 7, 4), customer_bucket_4=None)",1
"Row(order_ts_day=datetime.date(2026, 7, 5), customer_bucket_4=3)",1
